# Validate reconstruction of seasonal windows

In [ ]:
from __future__ import annotations

import math
import pickle
from collections import Counter
from pathlib import Path

import dill
import numpy as np
import pandas as pd

from os import environ

configured_root = environ.get("ML4GM_ROOT")
ROOT = Path(configured_root).expanduser().resolve() if configured_root else Path.cwd().resolve()
if not configured_root:
    if (ROOT / "01_Dataset.ipynb").is_file():
        ROOT = ROOT.parent
    if not (ROOT / "code" / "01_Dataset.ipynb").is_file():
        raise RuntimeError("Start this notebook from the release root or its code directory, or set ML4GM_ROOT.")
elif not ROOT.is_dir():
    raise FileNotFoundError(f"The configured ML4GM_ROOT directory does not exist: {ROOT}")

DATA_DIR = ROOT / "data" / "processed" / "seasonal_training"
ERA_LONG_CSV = ROOT / "data" / "raw" / "era5land_gee" / "wgms_global_era5land_1979_2025.csv"
MODEL_DIR = ROOT / "models" / "annual_seasonal"
OUT_DIR = ROOT / "results" / "analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

K_NEIGHBORS = 5
NN_VOTE_THRESHOLD = 0.5
REGION_ENSEMBLE_TOPK = 3
REGIONAL_MAIN_TEXT_MIN_N = 20

STATE_VARS = ["t2m", "rsn", "sde", "tsn", "fal", "sp"]
FLOW_VARS = ["sf", "smlt", "slhf", "ssr", "strd", "sshf", "ssrd", "tp"]
CLIMATE_VARS = STATE_VARS + FLOW_VARS
TARGET_BASE_SIGMA = {"winter": 0.58, "summer": 0.62}

OUTPUTS = {
    "records": OUT_DIR / "seasonal_window_withheld_validation_records.csv",
    "summary": OUT_DIR / "seasonal_window_withheld_validation_summary.csv",
    "regional": OUT_DIR / "seasonal_window_withheld_validation_regional.csv",
    "balance": OUT_DIR / "seasonal_window_balance_sensitivity.csv",
}

In [ ]:
def window_key_from_row(row) -> tuple[int, int, int, int] | None:
    vals = [
        row["season_start_month"],
        row["season_end_month"],
        row["season_start_year_offset"],
        row["season_end_year_offset"],
    ]
    if any(pd.isna(v) for v in vals):
        return None
    return (int(vals[0]), int(vals[1]), int(vals[2]), int(vals[3]))


def window_pair_label(key: tuple[int, int, int, int] | None) -> str | float:
    if key is None:
        return np.nan
    sm, em, so, eo = key
    return f"{so:+d}:{sm:02d}->{eo:+d}:{em:02d}"


def adjusted_month_index(month: int, offset: int) -> int:
    return int(offset) * 12 + int(month)


def key_start_index(key):
    return adjusted_month_index(key[0], key[2])


def key_end_index(key):
    return adjusted_month_index(key[1], key[3])


def key_duration(key) -> float:
    start = key_start_index(key)
    end = key_end_index(key)
    return float(end - start + 1) if end >= start else np.nan


def key_month_set(key) -> set[int]:
    start = key_start_index(key)
    end = key_end_index(key)
    if end < start:
        return set()
    return set(range(start, end + 1))


def month_set_jaccard(a, b) -> float:
    aset = key_month_set(a)
    bset = key_month_set(b)
    if not aset and not bset:
        return np.nan
    union = aset | bset
    return float(len(aset & bset) / len(union)) if union else np.nan


def key_distance_metrics(true_key, pred_key) -> dict[str, float | bool]:
    if true_key is None or pred_key is None:
        return {
            "exact_key_match": False,
            "start_month_abs_error": np.nan,
            "end_month_abs_error": np.nan,
            "duration_abs_error": np.nan,
            "month_set_jaccard": np.nan,
            "near_match": False,
        }
    start_err = abs(key_start_index(pred_key) - key_start_index(true_key))
    end_err = abs(key_end_index(pred_key) - key_end_index(true_key))
    duration_err = abs(key_duration(pred_key) - key_duration(true_key))
    jaccard = month_set_jaccard(true_key, pred_key)
    return {
        "exact_key_match": pred_key == true_key,
        "start_month_abs_error": float(start_err),
        "end_month_abs_error": float(end_err),
        "duration_abs_error": float(duration_err),
        "month_set_jaccard": jaccard,
        "near_match": bool((start_err <= 1 and end_err <= 1) or (pd.notna(jaccard) and jaccard >= 0.8)),
    }


def dominant_key(keys: list[tuple[int, int, int, int]]) -> tuple[tuple[int, int, int, int] | None, float]:
    if not keys:
        return None, np.nan
    counts = Counter(keys)
    top_count = max(counts.values())
    top_keys = sorted([key for key, count in counts.items() if count == top_count])
    return top_keys[0], float(top_count / sum(counts.values()))


def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 6371.0 * 2.0 * np.arcsin(np.sqrt(a))

In [ ]:
def load_observed_windows() -> dict[str, pd.DataFrame]:
    observed = {}
    for season in ["winter", "summer"]:
        path = DATA_DIR / f"wgms_global_{season}_era5land_1979_2025.csv"
        df = pd.read_csv(path, low_memory=False)
        for col in ["season_start_month", "season_end_month", "season_start_year_offset", "season_end_year_offset", "year"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df["season"] = season
        df["record_index"] = np.arange(len(df), dtype=int)
        df["record_id"] = season + "_" + df["record_index"].astype(str)
        df["true_window_key"] = df.apply(window_key_from_row, axis=1)
        df["true_window_pair"] = df["true_window_key"].map(window_pair_label)
        df = df.loc[df["true_window_key"].notna()].copy()
        observed[season] = df.reset_index(drop=True)
    return observed


def build_glacier_defaults(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for rgiid, group in df.groupby("RGIId", sort=False):
        keys = group["true_window_key"].dropna().tolist()
        key, share = dominant_key(keys)
        n = len(keys)
        if n < 3 or key is None:
            continue
        rows.append({
            "RGIId": rgiid,
            "n_obs_windows": int(n),
            "window_key": key,
            "window_pair": window_pair_label(key),
            "window_vote_share": share,
            "window_confidence": "high" if n >= 5 else "medium",
            "CenLon": float(pd.to_numeric(group["CenLon"], errors="coerce").median()),
            "CenLat": float(pd.to_numeric(group["CenLat"], errors="coerce").median()),
            "O1Region": group["O1Region"].dropna().iloc[0] if group["O1Region"].notna().any() else np.nan,
        })
    return pd.DataFrame(rows)


def region_counter(df: pd.DataFrame) -> dict[object, Counter]:
    counters: dict[object, Counter] = {}
    for o1, group in df.groupby("O1Region", dropna=False, sort=False):
        counters[o1] = Counter(group["true_window_key"].dropna().tolist())
    return counters


def glacier_key_counter(df: pd.DataFrame) -> dict[str, Counter]:
    counters: dict[str, Counter] = {}
    for rgiid, group in df.groupby("RGIId", sort=False):
        counters[rgiid] = Counter(group["true_window_key"].dropna().tolist())
    return counters


def top_region_windows(base_counter: Counter, subtract: Counter | None = None) -> list[dict]:
    counts = Counter(base_counter)
    if subtract:
        for key, value in subtract.items():
            counts[key] -= value
            if counts[key] <= 0:
                counts.pop(key, None)
    total = sum(counts.values())
    if total <= 0:
        return []
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:REGION_ENSEMBLE_TOPK]
    return [
        {
            "window_key": key,
            "window_pair": window_pair_label(key),
            "window_share": float(count / total),
            "window_count": int(count),
            "total_windows": int(total),
        }
        for key, count in ordered
    ]


def nearest_neighbor_window(row: pd.Series, defaults: pd.DataFrame):
    if defaults.empty or pd.isna(row["CenLon"]) or pd.isna(row["CenLat"]):
        return None
    pool = defaults.loc[
        defaults["O1Region"].eq(row["O1Region"])
        & defaults["CenLon"].notna()
        & defaults["CenLat"].notna()
        & defaults["RGIId"].ne(row["RGIId"])
    ].copy()
    if pool.empty:
        return None
    pool["distance_km"] = haversine_km(
        float(row["CenLon"]),
        float(row["CenLat"]),
        pool["CenLon"].astype(float),
        pool["CenLat"].astype(float),
    )
    nn = pool.nsmallest(K_NEIGHBORS, "distance_km").copy()
    nn["weight"] = 1.0 / (nn["distance_km"] + 1.0)
    votes = nn.groupby("window_key")["weight"].sum().sort_values(ascending=False)
    if votes.empty:
        return None
    top_key = votes.index[0]
    vote_share = float(votes.iloc[0] / votes.sum())
    return {
        "window_key": top_key,
        "window_vote_share": vote_share,
        "neighbor_count": int(len(nn)),
        "neighbor_distance_median_km": float(nn["distance_km"].median()),
    }

In [ ]:
def infer_window_for_row(row, season_df, defaults_full, region_counts, glacier_counts, regime):
    true_key = row["true_window_key"]
    subtract_region = Counter()

    if regime == "glacier_year_withheld":
        same_glacier = season_df.loc[season_df["RGIId"].eq(row["RGIId"]) & season_df["record_id"].ne(row["record_id"])]
        history_keys = same_glacier["true_window_key"].dropna().tolist()
        key, share = dominant_key(history_keys)
        n_history = len(history_keys)
        if n_history >= 3 and key is not None:
            return {
                "inferred_source": "glacier_history",
                "inferred_confidence": "high" if n_history >= 5 else "medium",
                "predicted_key": key,
                "candidate_keys": [key],
                "window_vote_share": share,
                "neighbor_count": 0,
                "neighbor_distance_median_km": np.nan,
                "window_ensemble_size": 1,
                "remaining_history_windows": n_history,
            }
        subtract_region[true_key] += 1
    elif regime == "glacier_held_out":
        subtract_region.update(glacier_counts.get(row["RGIId"], Counter()))
        n_history = 0
    else:
        raise ValueError(f"Unknown validation regime: {regime}")

    nn = nearest_neighbor_window(row, defaults_full)
    if nn is not None and nn["window_vote_share"] >= NN_VOTE_THRESHOLD:
        return {
            "inferred_source": "nearest_neighbor",
            "inferred_confidence": "low",
            "predicted_key": nn["window_key"],
            "candidate_keys": [nn["window_key"]],
            "window_vote_share": nn["window_vote_share"],
            "neighbor_count": nn["neighbor_count"],
            "neighbor_distance_median_km": nn["neighbor_distance_median_km"],
            "window_ensemble_size": 1,
            "remaining_history_windows": n_history,
        }

    topk = top_region_windows(region_counts.get(row["O1Region"], Counter()), subtract=subtract_region)
    if topk:
        keys = [entry["window_key"] for entry in topk]
        return {
            "inferred_source": "o1region_ensemble",
            "inferred_confidence": "very_low",
            "predicted_key": keys[0],
            "candidate_keys": keys,
            "window_vote_share": topk[0]["window_share"],
            "neighbor_count": 0,
            "neighbor_distance_median_km": np.nan,
            "window_ensemble_size": len(keys),
            "remaining_history_windows": n_history,
        }

    return {
        "inferred_source": "unavailable",
        "inferred_confidence": "unavailable",
        "predicted_key": None,
        "candidate_keys": [],
        "window_vote_share": np.nan,
        "neighbor_count": 0,
        "neighbor_distance_median_km": np.nan,
        "window_ensemble_size": 0,
        "remaining_history_windows": n_history,
    }


def validation_records_for_season(season: str, season_df: pd.DataFrame) -> pd.DataFrame:
    defaults_full = build_glacier_defaults(season_df)
    region_counts = region_counter(season_df)
    glacier_counts = glacier_key_counter(season_df)
    rows = []

    for regime in ["glacier_year_withheld", "glacier_held_out"]:
        for row in season_df.itertuples(index=False):
            row_s = pd.Series(row._asdict())
            inferred = infer_window_for_row(row_s, season_df, defaults_full, region_counts, glacier_counts, regime)
            true_key = row_s["true_window_key"]
            pred_key = inferred["predicted_key"]
            metrics = key_distance_metrics(true_key, pred_key)
            candidate_keys = inferred["candidate_keys"]
            ensemble_top3_hit = bool(true_key in candidate_keys) if candidate_keys else False
            if candidate_keys:
                candidate_errors = [
                    np.nanmean([
                        abs(key_start_index(k) - key_start_index(true_key)),
                        abs(key_end_index(k) - key_end_index(true_key)),
                    ])
                    for k in candidate_keys
                ]
                ensemble_best_month_error = float(np.nanmin(candidate_errors))
            else:
                ensemble_best_month_error = np.nan

            rows.append({
                "season": season,
                "validation_regime": regime,
                "record_id": row_s["record_id"],
                "RGIId": row_s["RGIId"],
                "year": int(row_s["year"]),
                "O1Region": row_s["O1Region"],
                "Area": row_s.get("Area", np.nan),
                "true_window_key": str(true_key),
                "true_window_pair": window_pair_label(true_key),
                "predicted_window_key": str(pred_key) if pred_key is not None else np.nan,
                "predicted_window_pair": window_pair_label(pred_key),
                "candidate_window_pairs": ";".join([str(window_pair_label(k)) for k in candidate_keys]),
                "ensemble_top3_hit": ensemble_top3_hit,
                "ensemble_best_month_error": ensemble_best_month_error,
                **metrics,
                **{k: v for k, v in inferred.items() if k not in {"predicted_key", "candidate_keys"}},
            })
    return pd.DataFrame(rows)

In [ ]:
def month_sequence(key: tuple[int, int, int, int], balance_year: int) -> list[tuple[int, int]]:
    sm, em, so, eo = key
    start_period = pd.Period(year=int(balance_year) + int(so), month=int(sm), freq="M")
    end_period = pd.Period(year=int(balance_year) + int(eo), month=int(em), freq="M")
    if end_period < start_period:
        return []
    return [(int(p.year), int(p.month)) for p in pd.period_range(start_period, end_period, freq="M")]


def load_era_lookup() -> dict[tuple[str, int, int], dict[str, float]]:
    usecols = ["RGIId", "year", "month"] + CLIMATE_VARS
    era = pd.read_csv(ERA_LONG_CSV, usecols=usecols, low_memory=False)
    era["year"] = pd.to_numeric(era["year"], errors="coerce").astype(int)
    era["month"] = pd.to_numeric(era["month"], errors="coerce").astype(int)
    return {
        (row.RGIId, int(row.year), int(row.month)): {var: getattr(row, var) for var in CLIMATE_VARS}
        for row in era.itertuples(index=False)
    }


def aggregate_key_for_record(row, key, era_lookup):
    values = []
    for year_month in month_sequence(key, int(row["year"])):
        lookup_key = (row["RGIId"], year_month[0], year_month[1])
        if lookup_key in era_lookup:
            values.append(era_lookup[lookup_key])
    if not values:
        return {var: np.nan for var in CLIMATE_VARS}
    frame = pd.DataFrame(values)
    out = {var: frame[var].mean() for var in STATE_VARS}
    out.update({var: frame[var].sum(min_count=1) for var in FLOW_VARS})
    return out


def feature_frame(row, key, feature_columns, era_lookup):
    values = {col: row.get(col, np.nan) for col in feature_columns}
    values.update(aggregate_key_for_record(row, key, era_lookup))
    return pd.DataFrame([values], columns=feature_columns).astype(float)


def load_model_and_features(season):
    with (MODEL_DIR / f"preprocessed_{season}_data.pkl").open("rb") as handle:
        payload = dill.load(handle)
    with (MODEL_DIR / f"xgboost_{season}_global.pkl").open("rb") as handle:
        model = pickle.load(handle)
    return model, list(payload["feature_columns"])


def predict_for_keys(row, keys, model, feature_columns, era_lookup):
    predictions = []
    for key in keys:
        X = feature_frame(row, key, feature_columns, era_lookup)
        # Use the same complete predictor requirement as seasonal reconstruction.
        if X.isna().any(axis=None):
            predictions.append(np.nan)
        else:
            predictions.append(float(model.predict(X)[0]))
    return predictions


def add_balance_sensitivity(records: pd.DataFrame, observed: dict[str, pd.DataFrame]) -> pd.DataFrame:
    era_lookup = load_era_lookup()
    enriched = []
    obs_lookup = {
        season: df.set_index("record_id", drop=False)
        for season, df in observed.items()
    }
    for season in ["winter", "summer"]:
        model, feature_columns = load_model_and_features(season)
        sub = records.loc[records["season"].eq(season)].copy()
        for rec in sub.itertuples(index=False):
            base_row = obs_lookup[season].loc[rec.record_id].to_dict()
            true_key = base_row["true_window_key"]
            if rec.inferred_source == "unavailable":
                true_pred = np.nan
                inferred_pred = np.nan
            else:
                true_pred = predict_for_keys(base_row, [true_key], model, feature_columns, era_lookup)[0]
                # Candidate keys are kept in memory to avoid parsing CSV-safe tuple strings.
                candidate_keys = INFERRED_KEY_MAP[(season, rec.validation_regime, rec.record_id)]
                inferred_predictions = predict_for_keys(base_row, candidate_keys, model, feature_columns, era_lookup)
                valid_predictions = [prediction for prediction in inferred_predictions if np.isfinite(prediction)]
                inferred_pred = float(np.mean(valid_predictions)) if valid_predictions else np.nan
            row = rec._asdict()
            row["pred_true_window"] = true_pred
            row["pred_inferred_window"] = inferred_pred
            row["delta_pred"] = inferred_pred - true_pred if pd.notna(inferred_pred) and pd.notna(true_pred) else np.nan
            row["abs_delta_pred"] = abs(row["delta_pred"]) if pd.notna(row["delta_pred"]) else np.nan
            row["target_base_sigma"] = TARGET_BASE_SIGMA[season]
            enriched.append(row)
    return pd.DataFrame(enriched)

In [ ]:
observed = load_observed_windows()
records_by_season = []
INFERRED_KEY_MAP = {}

for season, df in observed.items():
    season_records = validation_records_for_season(season, df)
    records_by_season.append(season_records)
    # Build an in-memory key map to avoid lossy tuple parsing after CSV-safe serialization.
    defaults_full = build_glacier_defaults(df)
    region_counts = region_counter(df)
    glacier_counts = glacier_key_counter(df)
    for regime in ["glacier_year_withheld", "glacier_held_out"]:
        for row in df.itertuples(index=False):
            row_s = pd.Series(row._asdict())
            inferred = infer_window_for_row(row_s, df, defaults_full, region_counts, glacier_counts, regime)
            INFERRED_KEY_MAP[(season, regime, row_s["record_id"])] = inferred["candidate_keys"]

records = pd.concat(records_by_season, ignore_index=True)
records = add_balance_sensitivity(records, observed)
records.to_csv(OUTPUTS["records"], index=False)
records.head()

In [ ]:
def summarize_group(group: pd.DataFrame, keys: list[str], summary_level: str) -> dict:
    delta = group["delta_pred"].dropna()
    abs_delta = group["abs_delta_pred"].dropna()
    base = group.loc[abs_delta.index, "target_base_sigma"]
    out = {key: group[key].iloc[0] for key in keys}
    out.update({
        "summary_level": summary_level,
        "n_records": int(len(group)),
        "n_balance_sensitivity": int(delta.shape[0]),
        "exact_match_rate": float(group["exact_key_match"].mean()) if len(group) else np.nan,
        "near_match_rate": float(group["near_match"].mean()) if len(group) else np.nan,
        "ensemble_top3_hit_rate": float(group["ensemble_top3_hit"].mean()) if len(group) else np.nan,
        "mean_start_month_abs_error": float(group["start_month_abs_error"].mean()),
        "median_start_month_abs_error": float(group["start_month_abs_error"].median()),
        "mean_end_month_abs_error": float(group["end_month_abs_error"].mean()),
        "median_end_month_abs_error": float(group["end_month_abs_error"].median()),
        "mean_duration_abs_error": float(group["duration_abs_error"].mean()),
        "mean_month_set_jaccard": float(group["month_set_jaccard"].mean()),
        "delta_pred_bias": float(delta.mean()) if len(delta) else np.nan,
        "delta_pred_mae": float(abs_delta.mean()) if len(abs_delta) else np.nan,
        "delta_pred_rmse": float(np.sqrt(np.mean(delta ** 2))) if len(delta) else np.nan,
        "abs_delta_pred_median": float(abs_delta.median()) if len(abs_delta) else np.nan,
        "abs_delta_pred_p90": float(abs_delta.quantile(0.9)) if len(abs_delta) else np.nan,
        "share_abs_delta_gt_0_10": float((abs_delta > 0.10).mean()) if len(abs_delta) else np.nan,
        "share_abs_delta_gt_0_25": float((abs_delta > 0.25).mean()) if len(abs_delta) else np.nan,
        "share_abs_delta_gt_base_sigma": float((abs_delta > base).mean()) if len(abs_delta) else np.nan,
    })
    return out

summary_rows = []
for keys, label in [
    (["season", "validation_regime"], "overall"),
    (["season", "validation_regime", "inferred_source"], "source"),
    (["season", "validation_regime", "inferred_confidence"], "confidence"),
]:
    for _, group in records.groupby(keys, dropna=False, sort=False):
        summary_rows.append(summarize_group(group, keys, label))
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUTS["summary"], index=False)

regional_rows = []
for keys, group in records.groupby(["season", "validation_regime", "O1Region"], dropna=False, sort=False):
    row = summarize_group(group, ["season", "validation_regime", "O1Region"], "regional")
    row["main_text_supported"] = bool(row["n_records"] >= REGIONAL_MAIN_TEXT_MIN_N)
    regional_rows.append(row)
regional = pd.DataFrame(regional_rows)
regional.to_csv(OUTPUTS["regional"], index=False)

balance_rows = []
for keys, group in records.groupby(["season", "validation_regime", "inferred_source", "inferred_confidence"], dropna=False, sort=False):
    balance_rows.append(summarize_group(group, ["season", "validation_regime", "inferred_source", "inferred_confidence"], "balance_sensitivity"))
balance = pd.DataFrame(balance_rows)
balance.to_csv(OUTPUTS["balance"], index=False)

summary

In [ ]:
for name, path in OUTPUTS.items():
    print(f"{name}: {path.relative_to(ROOT)}")

print("Records by season and regime:")
print(records.groupby(["season", "validation_regime"]).size())

print("Overall validation summary:")
print(summary.loc[summary["summary_level"].eq("overall"), [
    "season", "validation_regime", "n_records", "exact_match_rate", "near_match_rate",
    "mean_start_month_abs_error", "mean_end_month_abs_error", "delta_pred_mae", "abs_delta_pred_p90"
]])